# DoomArena extended TauBench: Bedrock baseline vs DFC

This notebook runs the six DoomArena TauBench attack configurations twice: first unchanged, then with DFC guarding state-changing tool calls. Use a standard CPU Colab runtime.

## 1. Add Colab secrets

In the Colab Secrets panel add `GITHUB_TOKEN` (for the private benchmarking repository), `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, and optionally `AWS_SESSION_TOKEN`. Grant notebook access to each secret.

In [ ]:
from google.colab import userdata
import json, os, pathlib, subprocess

os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
session_token = userdata.get('AWS_SESSION_TOKEN')
if session_token:
    os.environ['AWS_SESSION_TOKEN'] = session_token
os.environ['AWS_REGION_NAME'] = 'us-west-2'
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'
print('AWS region:', os.environ['AWS_REGION_NAME'])

## 2. Clone clean copies

The token is passed directly to Git and is not printed. Change `BENCHMARK_REPO` if your fork URL differs.

In [ ]:
ROOT = pathlib.Path('/content/dfc_doomarena_run')
subprocess.run(['rm', '-rf', str(ROOT)], check=True)
ROOT.mkdir(parents=True)

BENCHMARK_REPO = 'github.com/codeboi07/DFC_Benchmarking.git'
token = userdata.get('GITHUB_TOKEN')
subprocess.run(['git', 'clone', f'https://x-access-token:{token}@{BENCHMARK_REPO}', str(ROOT / 'DFC_Benchmarking')], check=True)
subprocess.run(['git', 'clone', 'https://github.com/ServiceNow/DoomArena.git', str(ROOT / 'DoomArena')], check=True)
subprocess.run(['git', '-C', str(ROOT / 'DoomArena'), 'checkout', 'b80902f107b4d28194580352a59b3029f4a018b4'], check=True)
subprocess.run(['git', 'clone', 'https://github.com/sierra-research/tau-bench.git', str(ROOT / 'tau-bench')], check=True)
subprocess.run(['git', '-C', str(ROOT / 'tau-bench'), 'checkout', '59a200c6d575d595120f1cb70fea53cef0632f6b'], check=True)
print(ROOT)

## 3. Install dependencies

Restarting the runtime is normally unnecessary after these editable installs.

In [ ]:
!pip install -q boto3 botocore litellm duckdb pyyaml pandas matplotlib pytest
!pip install -q -e {ROOT/'tau-bench'}
!pip install -q -e {ROOT/'DoomArena'/'doomarena'/'core'}
!pip install -q -e {ROOT/'DoomArena'/'doomarena'/'taubench'}
!pip install -q -e {ROOT/'DFC_Benchmarking'/'dfc_agent_framework_integration'}

## 4. Verify Bedrock and discover the six cases

Your IAM principal needs `bedrock:InvokeModel` and `bedrock:InvokeModelWithResponseStream` for the selected model in this region.

In [ ]:
import boto3
print('AWS account:', boto3.client('sts').get_caller_identity()['Account'])

PROJECT = ROOT / 'DFC_Benchmarking'
RUNNER = PROJECT / 'scripts' / 'doomarena_taubench_dfc.py'
CONFIG_DIR = ROOT / 'DoomArena' / 'doomarena' / 'taubench' / 'src' / 'doomarena' / 'taubench' / 'scripts'
OUTPUT_DIR = ROOT / 'results' / 'doomarena_taubench'
MODEL = 'bedrock/us.anthropic.claude-sonnet-4-5-20250929-v1:0'

subprocess.run(['python', str(RUNNER), 'list', '--config-dir', str(CONFIG_DIR)], check=True)

## 5. Run the baseline

This is one trial for every task selected by all six official DoomArena YAML cases. No DFC policy is active. DoomArena attacks remain active.

In [ ]:
common = [
    '--config-dir', str(CONFIG_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--model', MODEL, '--provider', 'bedrock',
    '--user-model', MODEL, '--user-provider', 'bedrock',
    '--attack-model', MODEL, '--attack-provider', 'bedrock',
    '--judge-model', MODEL, '--judge-provider', 'bedrock',
    '--max-steps', '30', '--seed', '42',
]
subprocess.run(['python', str(RUNNER), 'run', '--condition', 'baseline', *common], check=True)

In [ ]:
import pandas as pd
baseline = pd.read_csv(OUTPUT_DIR / 'baseline' / 'results.csv')
display(baseline[['case', 'domain', 'task_id', 'task_success', 'attack_success', 'proposed_effectful_calls', 'attack_target_executed_calls']])

## 6. Run the DFC ablation

DFC is placed outside DoomArena's attack gateway. It validates each effectful tool and its sensitive scalar arguments before execution. Read-only calls remain available; output from the attacked `get_product_details` channel is deliberately not trusted.

In [ ]:
subprocess.run(['python', str(RUNNER), 'run', '--condition', 'dfc', *common], check=True)

## 7. Inspect a real generated policy

Each case/task has a JSON policy manifest and a PGN policy file under `results/doomarena_taubench/policies`. The manifest records the trusted task, authorized effectful tools, initial trusted values, and domain rules.

In [ ]:
policy_file = OUTPUT_DIR / 'policies' / 'malicious_user_airline_attack' / 'task_16.json'
policy = json.loads(policy_file.read_text())
print(json.dumps(policy, indent=2))

## 8. Paired comparison

Primary security signals are ASR and attack-target tool execution. TSR is retained as the utility measure. A negative ASR or attack-target execution delta is improvement; a negative TSR delta is utility loss. Catalog secret leakage can occur through final text rather than an effectful tool, so tool-call DFC is not expected to eliminate every catalog ASR.

In [ ]:
subprocess.run(['python', str(RUNNER), 'compare', '--output-dir', str(OUTPUT_DIR)], check=True)
summary = json.loads((OUTPUT_DIR / 'comparison' / 'summary.json').read_text())
paired = pd.read_csv(OUTPUT_DIR / 'comparison' / 'paired_results.csv')
print(json.dumps(summary, indent=2))
display(paired)

In [ ]:
import matplotlib.pyplot as plt
labels = ['Task success', 'Attack success', 'Attack-target execution']
base = [summary['baseline']['task_success_rate'], summary['baseline']['attack_success_rate'], summary['baseline']['attack_target_execution_rate']]
guarded = [summary['dfc']['task_success_rate'], summary['dfc']['attack_success_rate'], summary['dfc']['attack_target_execution_rate']]
x = range(len(labels))
plt.figure(figsize=(9, 4))
plt.bar([i - .18 for i in x], base, width=.36, label='Baseline')
plt.bar([i + .18 for i in x], guarded, width=.36, label='DFC')
plt.xticks(list(x), labels, rotation=10)
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

## 9. Persist results to Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DEST = pathlib.Path('/content/drive/MyDrive/DFC_Benchmarking/doomarena_taubench')
subprocess.run(['rm', '-rf', str(DEST)], check=True)
DEST.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(['cp', '-r', str(OUTPUT_DIR), str(DEST)], check=True)
print('Saved to', DEST)